# 04 — Overall Evaluation Criterion and the Metrics Taxonomy
**Prerequisites:** `01_ab_testing_fundamentals.ipynb`.
**Connects to:** `05_guardrail_metrics.ipynb` (guardrails get the full treatment there);
`09_segmentation_heterogeneity.ipynb` (segment cuts of these same metrics).

## Narrative thread
```
The OEC concept (Kohavi) -> primary / secondary / guardrail taxonomy (overview)
   -> tradeoffs & metric gaming (Goodhart's law) -> worked metrics tree for a checkout funnel
```

## The Overall Evaluation Criterion (OEC)

Kohavi, Tang & Xu (2020) coined the **OEC**: the single metric (or small weighted set) an
organization commits to *before* running an experiment as the criterion for "did this ship." The
OEC should proxy long-term business value, not be trivially gameable in the short run, and be
measurable within the experiment window. A classic failure mode is optimizing a *proxy* metric
(clicks, session count) that diverges from the *true* long-term objective (retention, revenue,
user trust) — this is why the OEC is chosen deliberately and rarely equals "whatever moved."

Kohavi's own example: at Bing, ad clicks look like a good short-term OEC, but *degraded search
quality* can increase ad clicks (users can't find organic results, so they click ads out of
frustration) while destroying long-term trust and usage. The OEC has to encode the *durable*
outcome the business cares about, sometimes by combining metrics.

## Metric taxonomy

| Tier | Purpose | Statistical treatment |
|---|---|---|
| **Primary / OEC** | The ship/no-ship decision metric(s), pre-registered | Full power analysis (notebook 03), no multiplicity discount |
| **Secondary** | Diagnostic / supporting evidence, help interpret *why* the primary moved | Reported but rarely gate the decision alone; often many of them -> multiplicity matters (notebook 07) |
| **Guardrail** | Metrics that must *not* regress (latency, crashes, unsubscribe rate) | One-sided / non-inferiority tests — full treatment in `05_guardrail_metrics.ipynb` |

The taxonomy exists because not every metric should be treated the same statistically: a handful
of pre-registered primary metrics can use the full nominal $\alpha$, while dozens of secondary
metrics examined post hoc need correction (`07_multiple_testing.ipynb`), and guardrails need a
*non-inferiority* framing rather than a two-sided test (`05_guardrail_metrics.ipynb`).


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests

plt.rcParams.update({
    'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e8e8e8', 'axes.axisbelow': True,
    'axes.titlesize': 11, 'axes.titleweight': 'bold', 'legend.frameon': False,
})
np.random.seed(42)

In [2]:
# ── Worked metrics tree for an e-commerce checkout funnel ───────────────
funnel_stages = ['landing', 'add_to_cart', 'checkout_start', 'payment_info', 'purchase']
n_users = 100_000
np.random.seed(3)

# baseline conversion probability at each funnel stage (conditional on reaching it)
control_step_probs  = [1.00, 0.35, 0.70, 0.80, 0.85]
treatment_step_probs = [1.00, 0.37, 0.72, 0.79, 0.87]  # treatment: simplified checkout flow

def simulate_funnel(step_probs, n):
    remaining = n
    counts = []
    for p in step_probs:
        remaining = np.random.binomial(remaining, p)
        counts.append(remaining)
    return counts

control_counts = simulate_funnel(control_step_probs, n_users)
treatment_counts = simulate_funnel(treatment_step_probs, n_users)

funnel_df = pd.DataFrame({
    'stage': funnel_stages,
    'control': control_counts,
    'treatment': treatment_counts,
})
funnel_df['control_rate'] = funnel_df['control'] / n_users
funnel_df['treatment_rate'] = funnel_df['treatment'] / n_users
funnel_df['rel_lift'] = funnel_df['treatment_rate'] / funnel_df['control_rate'] - 1
print(funnel_df.to_string(index=False, float_format=lambda x: f'{x:.4f}'))

print("\nOEC candidate: purchase rate (end-to-end conversion) —",
      f"control={funnel_df.loc[4,'control_rate']:.4f}, treatment={funnel_df.loc[4,'treatment_rate']:.4f},",
      f"relative lift={funnel_df.loc[4,'rel_lift']:+.2%}")
print("Secondary metrics: add-to-cart rate and checkout_start rate help explain *why* purchase moved")
print("(here: simplified checkout raises add-to-cart & checkout_start, costs a little at payment_info,")
print(" nets out positive on the OEC — the secondary metrics tell the mechanism story).")

         stage  control  treatment  control_rate  treatment_rate  rel_lift
       landing   100000     100000        1.0000          1.0000    0.0000
   add_to_cart    35213      36861        0.3521          0.3686    0.0468
checkout_start    24689      26431        0.2469          0.2643    0.0706
  payment_info    19694      20837        0.1969          0.2084    0.0580
      purchase    16716      18183        0.1672          0.1818    0.0878

OEC candidate: purchase rate (end-to-end conversion) — control=0.1672, treatment=0.1818, relative lift=+8.78%
Secondary metrics: add-to-cart rate and checkout_start rate help explain *why* purchase moved
(here: simplified checkout raises add-to-cart & checkout_start, costs a little at payment_info,
 nets out positive on the OEC — the secondary metrics tell the mechanism story).


## Tradeoffs and Goodhart's law

> "When a measure becomes a target, it ceases to be a good measure." — Goodhart's law

If an OEC or, worse, a secondary metric used informally as a proxy target, is optimized directly
by teams (rather than by genuinely improving the user experience), it can be gamed: e.g.,
optimizing "time on page" can reward making a page *harder to use*; optimizing "notifications
sent" can reward spam. Guardrails against gaming:

- Pair every "engagement" style metric with a **quality/trust guardrail** (unsubscribe rate,
  complaint rate) — see `05_guardrail_metrics.ipynb`.
- Prefer OECs that are hard to move without genuinely helping the user (e.g., long-run retention
  over raw session count).
- Periodically re-validate that the OEC still correlates with the outcomes it's a proxy for
  (surveys, long-run holdouts — `10_uplift_holdouts.ipynb`).

## Key takeaways

| Concept | Statement |
|---|---|
| OEC | The pre-registered, hard-to-game metric (or small weighted combination) that decides ship/no-ship |
| Primary vs secondary vs guardrail | Different statistical treatment: full power for primary, multiplicity correction for many secondaries, non-inferiority for guardrails |
| Goodhart's law | A metric optimized directly stops measuring what it was meant to measure; pair OECs with guardrails |

## References

- Kohavi, R., Tang, D., & Xu, Y. (2020). *Trustworthy Online Controlled Experiments*, Ch. 6-7 (OEC and metrics).
- Deng, A., Lu, J., & Litz, S. Experimentation platform metrics design — Microsoft/LinkedIn experimentation literature (see Kohavi et al. 2020 for a full bibliography).
- See `05_guardrail_metrics.ipynb` for the statistical treatment of guardrail metrics.
